# Topic 6 — K-Nearest Neighbors (KNN)

> **Goal of this notebook:** understand the simplest "lazy" learner — classify a point by the majority vote of its nearest neighbours — and see how the choice of `k`, the distance metric, and feature scaling shape its behaviour.

**Contents**
1. Concept & intuition
2. The mathematics (distance metrics, voting)
3. Choosing k, scaling & the curse of dimensionality
4. The dataset: Iris
5. Training & the effect of k
6. Evaluation
7. Visualising the decision boundary
8. KNN from scratch
9. Pros, cons & when to use
10. Summary

## 1. Concept & Intuition

KNN makes no model during training — it just **stores the data** (hence "lazy"). To classify a new point it:
1. measures the distance to every training point,
2. finds the **k** closest ones,
3. takes a **majority vote** of their labels (for regression: the average).

The intuition: *similar inputs tend to have similar outputs.* Everything hinges on how we measure "similar" (the distance metric) and how many neighbours we consult (`k`).

## 2. The Mathematics

Distance between points $\mathbf{a}$ and $\mathbf{b}$ (the **Minkowski** family):

$$d(\mathbf{a}, \mathbf{b}) = \left( \sum_{i=1}^{n} |a_i - b_i|^p \right)^{1/p}$$

- $p=2$ → **Euclidean** distance (straight line).
- $p=1$ → **Manhattan** distance (grid steps).

Prediction = the most common label among the `k` nearest neighbours:

$$\hat{y} = \text{mode}\{ y_{(1)}, y_{(2)}, \dots, y_{(k)} \}$$

(optionally weighted by $1/d$ so closer neighbours count more).

## 3. Choosing k, Scaling & the Curse of Dimensionality

- **Small k** (e.g. 1) → low bias, high variance (noisy, overfits). **Large k** → smoother boundary, can underfit. A common heuristic is $k \approx \sqrt{n}$; better to tune by cross-validation.
- **Scaling is essential**: distances are dominated by large-range features unless you standardise.
- **Curse of dimensionality**: in high dimensions all points become roughly equidistant, so KNN degrades. Reduce dimensions (e.g. PCA) first if needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: Iris

The classic 150-sample dataset: 3 iris species described by 4 measurements (sepal/petal length & width). Small, clean, and ideal for visualising KNN.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
print('Shape:', X.shape, '| classes:', dict(zip(iris.target_names, np.bincount(y))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 5. Training & the Effect of k

We cross-validate accuracy across a range of `k` values to find the sweet spot.

In [ ]:
ks = range(1, 26)
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                            X_train_s, y_train, cv=5).mean() for k in ks]
best_k = ks[int(np.argmax(cv_scores))]

plt.plot(list(ks), cv_scores, marker='o')
plt.axvline(best_k, color='r', ls='--', label=f'best k = {best_k}')
plt.xlabel('k (number of neighbours)'); plt.ylabel('5-fold CV accuracy')
plt.title('Choosing k'); plt.legend(); plt.show()
print('Best k:', best_k)

## 6. Evaluation

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)
print('Test accuracy:', round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 7. Visualising the Decision Boundary

Using only the two petal features so we can plot in 2D.

In [ ]:
X2 = StandardScaler().fit_transform(X[:, 2:4])  # petal length & width
knn2 = KNeighborsClassifier(n_neighbors=best_k).fit(X2, y)

h = 0.02
x_min, x_max = X2[:, 0].min() - 1, X2[:, 0].max() + 1
y_min, y_max = X2[:, 1].min() - 1, X2[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
plt.scatter(X2[:, 0], X2[:, 1], c=y, cmap='viridis', edgecolor='k', s=25)
plt.xlabel('petal length (scaled)'); plt.ylabel('petal width (scaled)')
plt.title(f'KNN decision boundary (k={best_k})'); plt.show()

## 8. KNN From Scratch

Euclidean distance + majority vote — the entire algorithm in a few lines.

In [ ]:
class KNNScratch:
    def __init__(self, k=5):
        self.k = k
    def fit(self, X, y):
        self.X, self.y = np.asarray(X), np.asarray(y)
        return self
    def predict(self, X):
        preds = []
        for x in np.asarray(X):
            dists = np.sqrt(((self.X - x) ** 2).sum(axis=1))
            idx = np.argsort(dists)[:self.k]
            preds.append(Counter(self.y[idx]).most_common(1)[0][0])
        return np.array(preds)

scratch = KNNScratch(k=best_k).fit(X_train_s, y_train)
print('From-scratch accuracy:', round(accuracy_score(y_test, scratch.predict(X_test_s)), 4))
print('scikit-learn accuracy:', round(accuracy_score(y_test, y_pred), 4))

## 9. Pros, Cons & When to Use

**Pros**
- Dead simple, no training phase, naturally handles multi-class.
- Makes no assumption about the data's shape (non-parametric).
- Decision boundary can be arbitrarily complex.

**Cons**
- **Slow at prediction** — must compare against all training points.
- Memory-hungry (stores the whole dataset).
- Sensitive to scaling, irrelevant features, and the curse of dimensionality.

**When to use**
- Small/medium datasets with meaningful distances and low dimensionality.
- As a simple baseline, or for recommendation / similarity tasks.

## 10. Summary

- KNN classifies by the **majority vote of the k nearest points** — no training, just storage.
- The **distance metric** and **k** define its behaviour; small k overfits, large k underfits.
- **Scaling is mandatory** because the algorithm is distance-based.
- On Iris we tuned `k` by cross-validation, visualised the boundary, and matched scikit-learn with a from-scratch implementation.